# Self-supervised pretraining with AstroLens AstroPT

A tutorial-scale pretraining example for `astrolens.models.astropt.AstroPT`
(Smith et al., 2024, https://arxiv.org/abs/2405.14930), using
[`Smith42/galaxies`](https://huggingface.co/datasets/Smith42/galaxies) —
the same unlabeled dataset the reference authors pretrain and probe their
own released checkpoints on (streamed, first 4,000 examples; see
`utils/galaxies.py`).

Unlike `Linformer` and `GCNN`, AstroPT is a self-supervised backbone: it
learns from unlabeled images alone, with a causal next-patch prediction
objective (`model.loss(images)`). This notebook trains a small model from
scratch to demonstrate the mechanism and saves a checkpoint.

For downstream tasks, see
[`astropt_finetuning.ipynb`](astropt_finetuning.ipynb),
[`astropt_similarity_search.ipynb`](astropt_similarity_search.ipynb), and
[`astropt_anomaly_detection.ipynb`](astropt_anomaly_detection.ipynb), which
all use a released, much larger pretrained checkpoint from
[`Smith42/astroPT`](https://huggingface.co/Smith42/astroPT) instead of the
tiny from-scratch model trained here — five epochs on 4k images is enough to
see the objective decrease, not to reach a genuinely useful backbone.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [1]:
!pip install -q datasets torchvision

## Imports

In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import astrolens
from utils import galaxies

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load the dataset

Pretraining needs images only, no labels: stream the first `N_EXAMPLES`
from `Smith42/galaxies` and split 90/10 (`utils/galaxies.py`).

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 64
N_EXAMPLES = 4000

examples = galaxies.load_galaxies(N_EXAMPLES, columns=["image"])
train_idx, val_idx = galaxies.split_9010(examples)

# no pixel Normalize here: AstroPT.patchify normalizes each patch to zero
# mean/unit variance itself, matching the reference model's pretraining input
train_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(90),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
    ]
)

train_dataset = galaxies.GalaxiesDataset([examples[i] for i in train_idx], train_transform)
val_dataset = galaxies.GalaxiesDataset([examples[i] for i in val_idx], eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)

len(train_dataset), len(val_dataset)

Resolving data files:   0%|          | 0/1533 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1533 [00:00<?, ?it/s]

(3600, 400)

## Create the model

A small config (`patch_size=16` -> 196 patches at 224x224, `dim=192`,
`depth=6`, `heads=3`) so a few epochs run in reasonable time on one GPU.
`num_classes` is left unset: `forward()` then returns next-patch predictions,
and `loss()` computes the autoregressive (Huber) pretraining objective.

In [4]:
PATCH_SIZE = 16
DIM = 192
DEPTH = 6
HEADS = 3

model = astrolens.create_model(
    "astropt",
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    dim=DIM,
    depth=DEPTH,
    heads=HEADS,
).to(device)

sum(p.numel() for p in model.parameters())

2989248

## Pretrain

In [5]:
PRETRAIN_EPOCHS = 5
LR = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, count = 0.0, 0
    with torch.set_grad_enabled(train):
        for images in loader:
            images = images.to(device)
            loss = model.loss(images)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            count += images.size(0)

    return total_loss / count


for epoch in range(1, PRETRAIN_EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

epoch 01 train_loss=0.3725 val_loss=0.3648


epoch 02 train_loss=0.3198 val_loss=0.3487


epoch 03 train_loss=0.3078 val_loss=0.3424


epoch 04 train_loss=0.2997 val_loss=0.3365


epoch 05 train_loss=0.2935 val_loss=0.3318


## Save the checkpoint

In [6]:
torch.save(
    {
        "model": model.state_dict(),
        "img_size": IMG_SIZE,
        "patch_size": PATCH_SIZE,
        "dim": DIM,
        "depth": DEPTH,
        "heads": HEADS,
    },
    "astropt_galaxies_pretrained.pt",
)